# Mantenimiento Predictivo - Análisis Exploratorio de Datos (EDA)

Este notebook realiza el análisis exploratorio del **NASA C-MAPSS Turbofan Dataset**, con el objetivo de entender los datos de los sensores antes de entrenar nuestros modelos de Machine Learning.

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Añadir el directorio raíz al path para importar nuestros módulos
root_dir = Path.cwd().parent
sys.path.append(str(root_dir))

from ml.src.data.data_loader import load_dataset, add_rul_column

# Configuraciones de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("viridis")
pd.set_option('display.max_columns', 30)

## 1. Carga de Datos
Cargamos el subdataset **FD001** (Condiciones de operación: 1, Condiciones de falla: 1 - HPC Degradation).

In [ ]:
data_path = root_dir / 'data' / 'raw'
dataset = load_dataset(data_path, 'FD001')

train_df = dataset['train']
train_df = add_rul_column(train_df)

print(f"Dimensiones del dataset de entrenamiento: {train_df.shape}")
train_df.head()

## 2. Análisis del RUL (Remaining Useful Life)
Veamos cómo se distribuye el RUL máximo para cada motor en el dataset.

In [ ]:
max_rul = train_df.groupby('engine_id')['cycle'].max()

plt.figure(figsize=(10, 6))
sns.histplot(max_rul, bins=20, kde=True)
plt.title('Distribución de Ciclos de Vida (RUL máximo) por Motor')
plt.xlabel('Ciclos Totales')
plt.ylabel('Cantidad de Motores')
plt.show()

## 3. Comportamiento de Sensores vs RUL
Vamos a graficar algunos sensores de un motor específico para ver cómo cambian a medida que el motor se degrada (el RUL se acerca a 0).

In [ ]:
engine_1 = train_df[train_df['engine_id'] == 1]

sensors_to_plot = ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_11', 'sensor_12']

fig, axes = plt.subplots(3, 2, figsize=(15, 12))
fig.suptitle('Degradación de Sensores para el Motor 1', fontsize=16)
axes = axes.flatten()

for i, sensor in enumerate(sensors_to_plot):
    sns.lineplot(data=engine_1, x='cycle', y=sensor, ax=axes[i], color='crimson')
    axes[i].set_title(f'Lectura de {sensor} sobre el tiempo')
    axes[i].set_xlabel('Ciclo Operativo')
    axes[i].set_ylabel('Valor')

plt.tight_layout()
plt.show()